In [1]:
from importlib import import_module
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / '.data/annotated').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Open this notebook from inside the project.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

curation = import_module('src.steps.03_data_curation')
CuratedItem = curation.CuratedItem
build_dataset = curation.build_dataset
pair_paths = curation.pair_paths
push_to_hub = curation.push_to_hub
summarize = curation.summarize
write_dataset_card = curation.write_dataset_card
write_splits = curation.write_splits

TRAIN_FRACTION, VALIDATION_FRACTION, TEST_FRACTION, SEED = 0.85, 0.05, 0.10, 42


In [2]:
pairs = pair_paths()
print(f'Annotated pairs available: {len(pairs):,}')


Annotated pairs available: 14,635


In [3]:
splits, invalid = build_dataset(
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
    seed=SEED,
)
summary = summarize(splits)

for name in ('train', 'validation', 'test'):
    print(f"{name}: {summary[name]['count']:,} rows ({summary[name]['percentage']:.3f}%)")
print(f'invalid targets: {len(invalid)}')
for entry in invalid[:5]:
    print(entry)


train: 12,440 rows (85.002%)
validation: 732 rows (5.002%)
test: 1,463 rows (9.997%)
invalid targets: 0


In [ ]:
record = splits['test'][0].to_record()
print('resume_id:', record['resume_id'], '| category:', record['category'])
print('columns:', list(record))
print('resume_text (X) characters:', len(record['resume_text']))
print('target_json (Y) fields:', list(record['target_json']))


In [5]:
# Save the splits to .data/curated/ as train.jsonl, validation.jsonl, test.jsonl plus stats.json
paths = write_splits(splits)
write_dataset_card(summary)
for name, path in paths.items():
    print(f'{name} -> {path}')


train -> /Volumes/Code/Github/resume-parser-model/.data/curated/train.jsonl
validation -> /Volumes/Code/Github/resume-parser-model/.data/curated/validation.jsonl
test -> /Volumes/Code/Github/resume-parser-model/.data/curated/test.jsonl


In [6]:
# Upload the complete .data pipeline to the HF_DATASET_REPO in .env (needs HF_TOKEN)
dataset_url = push_to_hub()
dataset_url